In [22]:
import pandas as pd

df = pd.read_csv('data/train.csv')
#print(df.head())
#print(df.info())
#print(df.describe())
print(df.isnull().sum())

id                                     0
Age                                    0
Flight Distance                        0
Inflight wifi service                  0
Departure/Arrival time convenient      0
Ease of Online booking                 0
Gate location                          0
Food and drink                         0
Online boarding                        0
Seat comfort                           0
Inflight entertainment                 0
On-board service                       0
Leg room service                       0
Baggage handling                       0
Checkin service                        0
Cleanliness                            0
Departure Delay in Minutes             0
Arrival Delay in Minutes             292
Gender                                 0
Customer Type                          0
Type of Travel                         0
Class                                  0
satisfaction                           0
dtype: int64


In [23]:
print(df["Arrival Delay in Minutes"].describe())
print(df["Arrival Delay in Minutes"].isnull().sum())
df["Arrival Delay in Minutes"].value_counts()

count    699343.000000
mean          1.134280
std           5.749304
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max         491.000000
Name: Arrival Delay in Minutes, dtype: float64
292


Arrival Delay in Minutes
0.0      640415
1.0        4546
2.0        4141
3.0        3881
4.0        3612
          ...  
122.0         1
120.0         1
103.0         1
227.0         1
76.0          1
Name: count, Length: 168, dtype: int64

In [20]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import KNNImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, TargetEncoder

target_col = "satisfaction"
X = df.drop(columns=["id", target_col])
y = df[target_col]

categorical_cols = X.select_dtypes(include=["object", "category"]).columns
numeric_cols = X.select_dtypes(include="number").columns

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            TargetEncoder(
                target_type="binary", smooth="auto", cv=5, random_state=42
            ),
            categorical_cols,
        ),
        (
            "numeric",
            Pipeline(
                steps=[
                    ("scaler", RobustScaler()),
                    ("imputer", KNNImputer(n_neighbors=5, weights="distance")),
                ]
            ),
            numeric_cols,
        ),
    ]
 )

preprocessing_pipeline = Pipeline(steps=[("preprocessor", preprocessor)])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
 )

X_train_processed = preprocessing_pipeline.fit_transform(X_train, y_train)
X_test_processed = preprocessing_pipeline.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)

Processed training shape: (559708, 21)
Processed test shape: (139927, 21)


In [24]:
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

X_quick_train, _, y_quick_train, _ = train_test_split(
    X_train,
    y_train,
    train_size=min(20_000, len(X_train)),
    random_state=42,
    stratify=y_train,
 )

X_quick_valid = X_test.sample(
    n=min(10_000, len(X_test)), random_state=42
 )
y_quick_valid = y_test.loc[X_quick_valid.index]

quick_model = Pipeline(
    steps=[
        ("preprocessor", clone(preprocessor)),
        (
            "classifier",
            HistGradientBoostingClassifier(
                max_iter=100,
                max_depth=6,
                learning_rate=0.1,
                l2_regularization=1.0,
                max_bins=63,
                class_weight="balanced",
                random_state=42,
            ),
        ),
    ]
 )

quick_model.fit(X_quick_train, y_quick_train)
quick_predictions = quick_model.predict(X_quick_valid)
quick_probabilities = quick_model.predict_proba(X_quick_valid)[:, 1]

print("Quick validation accuracy:", accuracy_score(y_quick_valid, quick_predictions))
print("Quick validation ROC AUC:", roc_auc_score(y_quick_valid, quick_probabilities))

Quick validation accuracy: 0.915
Quick validation ROC AUC: 0.9524125780082351


In [25]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(quick_model, X, y, scoring='roc_auc', cv=5)
print(scores)
print(scores.mean())

[0.9576167  0.95675082 0.95716166 0.95615742 0.95710649]
0.9569586179661667


In [30]:
#taken from tune.py
params =  {
    "max_iter": 600,
    "max_depth": 8,
    "learning_rate": 0.043423793428711374,
    "l2_regularization": 0.05052609512159515,
    "max_bins": 211,
    "min_samples_leaf": 23,
    "class_weight": "balanced", 
    "random_state": 42
}

full_model = Pipeline(
    steps=[
        ("preprocessor", clone(preprocessor)),
        (
            "classifier",
            HistGradientBoostingClassifier(**params),
        ),
    ]
 )

#scores = cross_val_score(full_model, X, y, cv=5)
#print(scores)
full_model.fit(X, y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numeric', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the differen

In [31]:
df_test = pd.read_csv("data/test.csv")
preds = full_model.predict_proba(df_test)[:, 1]
submission = pd.DataFrame({"id": df_test["id"], "satisfaction": preds})
submission.to_csv("data/submission.csv", index=False)